# Pipeline 2: Text Generation

Text generation is the core task of **decoder-only** models (GPT family).
Given a prompt, the model predicts the next token, appends it, and repeats — this loop is called **autoregressive generation**.

Unlike encoder models (which read the whole sequence), a decoder only looks **left to right**: each token can only attend to previous tokens, never future ones.

In [1]:
from transformers import pipeline
from transformers import GenerationConfig


/home/v/Desktop/Projects/001-transformers-course/.env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load the environment variables

Load dotenv to be able to access the HuggingFace API token.

In [2]:
from dotenv import load_dotenv
import os

load_dotenv(".secrets")
hf_token = os.getenv("HF_TOKEN")

## Load the pipeline

We use `gpt2` — small (~500 MB), no token required, and good enough to see the architecture in action.

`max_new_tokens` controls how many tokens the model generates **on top of the prompt**.
Setting it avoids runaway generation.

In [3]:
generator = pipeline("text-generation", model="gpt2", hf_token=hf_token)

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 12297.33it/s]


## Basic example — greedy decoding

By default the model picks the **most probable token** at each step (greedy decoding).
It's fast and deterministic, but tends to produce repetitive or flat text.

In [4]:
generation_config = GenerationConfig(
    max_length=50,
    max_new_tokens=100,
)

generator("The goblin keep training daily in order to", generation_config=generation_config)

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[{'generated_text': 'The goblin keep training daily in order to keep the goblins from getting too close to the goblin keep.\n\nThe goblin keep is a place where goblins can be found.\n\nThe goblin keep is a place where goblins can be found.\n\nThe goblin keep is a place where goblins can be found.\n\nThe goblin keep is a place where goblins can be found.\n\nThe goblin keep is a place where goblins can be found.\n\nThe goblin keep is a place where goblins can be found.\n\nThe goblin'}]

## Sampling — adding randomness

With `do_sample=True` the model draws from the probability distribution instead of always picking the top token.
Two parameters shape how wild or conservative that distribution is:

- **`temperature`** — values below 1.0 make the model more confident (less random); above 1.0 make it more creative (more random).
- **`top_k`** — at each step, only consider the top-k most probable tokens. Limits incoherent leaps.

Try running the same prompt several times — you will get different outputs each time.

In [5]:
generation_config_confident = GenerationConfig(
    max_length=50,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.5,
    top_k=50,
)

generator("The goblin keep training daily in order to", generation_config=generation_config_confident)

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'The goblin keep training daily in order to make them stronger and stronger. They are very powerful, and they will always try to kill you.\n\nYou can use the following abilities to gain the ability to use the goblin keep training:\n\nAll the goblin keep training in order to make them stronger and stronger. They are very powerful, and they will always try to kill you. All the goblin keep training in order to make them stronger and stronger. They are very powerful, and they will always try to kill you. All the goblin'}]

In [6]:
generation_config_random = GenerationConfig(
    max_length=50,
    max_new_tokens=100,
    do_sample=True,
    temperature=1.5,
    top_k=50,
)

generator("The goblin keep training daily in order to", generation_config=generation_config_random)

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': "The goblin keep training daily in order to bring the beast back. Even now, many people use the train once there isn't a corpse remaining. It works against the village members which seems to be all around the world because the village guards get worried about these goblins while fighting them.The other villages are the other dwarves and others get killed every single day at their command. They have been known to kill even a man here once their numbers increased significantly with each death which has become a common fact. One goblin, Tenegis has not"}]

## Multiple sequences — `num_return_sequences`

The pipeline can generate several independent completions for the same prompt in one call.
Useful for picking the best result or exploring variation without calling the model in a loop.

In [7]:
generation_config = GenerationConfig(
    max_length=50,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.7,
    top_k=50,
    num_return_sequences=3,
)

generator("The goblin keep training daily in order to", generation_config=generation_config)

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': "The goblin keep training daily in order to be able to avoid that.\n\nBut if you're the kind of person who reads the story, you'll find that the goblins are quite good at it. They usually get to learn a few things, but they don't necessarily know how to use them.\n\nSo they're able to use a little bit of their skills to get around the barrier and become stronger.\n\nAnd that's what the Goblin Village is about. This is where the two kinds of monsters come in.\n"},
 {'generated_text': 'The goblin keep training daily in order to get an idea of when a goblin will enter the zone.\n\nGoblin Keep Training is a part of the goblin keep training system. The training is done by the goblin keeper, who will train a goblin by saying a word or two. The keeper can say anything from "yes", "yes", "no", "yes" or "no".\n\nThe goblin keeper is not a good trainer. He only looks after any goblin that has his orders. The goblin keeper is the one'},
 {'generated_text': 'The goblin keep tra

## Key takeaways

| Concept | What it means |
|---|---|
| Decoder-only | Tokens attend only to previous tokens — suited for generation, not classification |
| Autoregressive | Output is built one token at a time, each step conditioned on all previous ones |
| Greedy decoding | Always picks the most probable next token — deterministic but repetitive |
| `do_sample=True` | Draws from the distribution — introduces randomness and variety |
| `temperature` | Controls sharpness of the distribution: low = safe, high = creative |
| `top_k` | Restricts sampling to the k most likely tokens at each step |